In [4]:
#import pandas and numpy to apply to data
import pandas as pd
import numpy as np

#install uciimlrepo package to access the data
#%pip install ucimlrepo

#import the dataset
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
breast_cancer = fetch_ucirepo(id=14) 
  
# data (as pandas dataframes) 
X = breast_cancer.data.features 
y = breast_cancer.data.targets 
  
# metadata 
#print(breast_cancer.metadata) 
  
# variable information 
#print(breast_cancer.variables) 


In [7]:
#combine into a single dataframe
df = pd.concat([X, y], axis=1)
print(df)

#verify column name
#print(y.columns.tolist())
#Output was ['Class']

#melt
df_melted = pd.melt(
    df,
    id_vars=['Class'],
    var_name='Measurement',
    value_name='Value'
)
print(df_melted.head())

       age menopause tumor-size inv-nodes node-caps  deg-malig breast  \
0    30-39   premeno      30-34       0-2        no          3   left   
1    40-49   premeno      20-24       0-2        no          2  right   
2    40-49   premeno      20-24       0-2        no          2   left   
3    60-69      ge40      15-19       0-2        no          2  right   
4    40-49   premeno        0-4       0-2        no          2  right   
..     ...       ...        ...       ...       ...        ...    ...   
281  30-39   premeno      30-34       0-2        no          2   left   
282  30-39   premeno      20-24       0-2        no          3   left   
283  60-69      ge40      20-24       0-2        no          1  right   
284  40-49      ge40      30-34     5-Mar        no          3   left   
285  50-59      ge40      30-34     5-Mar        no          3   left   

    breast-quad irradiat                 Class  
0      left_low       no  no-recurrence-events  
1      right_up       no 

In [8]:
#pivot
#temporary sample ID column
df_melted['Sample_ID'] = df_melted.groupby(['Class', 'Measurement']).cumcount()

#pivot back to wide format
df_pivoted = df_melted.pivot(
    index=['Sample_ID', 'Class'],
    columns='Measurement',
    values='Value'
).reset_index()

#drop temporary sample ID column
df_pivoted = df_pivoted.drop(columns=['Sample_ID'])

print(df_pivoted.head())

Measurement                 Class    age breast breast-quad deg-malig  \
0            no-recurrence-events  30-39   left    left_low         3   
1               recurrence-events  50-59   left    left_low         2   
2            no-recurrence-events  40-49  right    right_up         2   
3               recurrence-events  40-49   left    left_low         1   
4            no-recurrence-events  40-49   left    left_low         2   

Measurement inv-nodes irradiat menopause node-caps tumor-size  
0                 0-2       no   premeno        no      30-34  
1                 0-2       no   premeno        no      15-19  
2                 0-2       no   premeno        no      20-24  
3                 0-2       no   premeno        no      40-44  
4                 0-2       no   premeno        no      20-24  


In [10]:
#aggregation
#Convert value to numeric
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')

#drop rows where value becomes NaN
df_melted = df_melted.dropna(subset=['Value'])

df_agg = df_melted.groupby(['Class', 'Measurement'])['Value'].agg(['mean', 'std', 'min', 'max']).reset_index()
print(df_agg.head())

                  Class Measurement      mean       std  min  max
0  no-recurrence-events   deg-malig  1.905473  0.697151  1.0  3.0
1     recurrence-events   deg-malig  2.388235  0.725293  1.0  3.0


In [11]:
#iteration
for row in df_agg.head(10).itertuples():
    print(f"Class {row.Class} has a mean {row.Measurement} of {row.mean:.2f}")

Class no-recurrence-events has a mean deg-malig of 1.91
Class recurrence-events has a mean deg-malig of 2.39


In [12]:
#groupby
#groupby class
df_by_class = df_agg.groupby('Class')['Measurement'].count().reset_index()
print(df_by_class)

                  Class  Measurement
0  no-recurrence-events            1
1     recurrence-events            1
